<a href="https://colab.research.google.com/github/bhargavi1973/machine-learning/blob/main/student_performance_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

In [ ]:
df = pd.read_csv('student_performance_prediction.csv')
df = df.head(1500)

In [ ]:
df

,Student ID,Study Hours per Week,Attendance Rate,Previous Grades,Participation in Extracurricular Activities,Parent Education Level,Passed
0,S00001,12.5,NaN,75.0,Yes,Master,Yes
1,S00002,9.3,95.3,60.6,No,High School,No
2,S00003,13.2,NaN,64.0,No,Associate,No
3,S00004,17.6,76.8,62.4,Yes,Bachelor,No
4,S00005,8.8,89.3,72.7,No,Master,No
...,...,...,...,...,...,...,...
1495,S01496,20.0,101.7,63.5,Yes,Doctorate,No
1496,S01497,20.3,100.1,68.9,No,High School,Yes
1497,S01498,16.0,57.0,56.9,NaN,Doctorate,Yes
1498,S01499,15.1,60.1,36.2,No,Associate,No


In [ ]:
print(f"Original DataFrame shape: {df.shape}")
df = df.dropna()
print(f"DataFrame shape after dropping missing values: {df.shape}")

Original DataFrame shape: (1500, 7)
DataFrame shape after dropping missing values: (1094, 7)


In [ ]:
df


,Student ID,Study Hours per Week,Attendance Rate,Previous Grades,Participation in Extracurricular Activities,Parent Education Level,Passed
1,S00002,9.3,95.3,60.6,No,High School,No
3,S00004,17.6,76.8,62.4,Yes,Bachelor,No
4,S00005,8.8,89.3,72.7,No,Master,No
5,S00006,8.8,73.8,69.3,Yes,High School,Yes
6,S00007,17.9,38.6,93.6,No,Doctorate,Yes
...,...,...,...,...,...,...,...
1493,S01494,10.9,87.3,76.0,Yes,Master,No
1494,S01495,17.7,87.9,39.4,No,Master,No
1495,S01496,20.0,101.7,63.5,Yes,Doctorate,No
1496,S01497,20.3,100.1,68.9,No,High School,Yes


In [ ]:
df = df.drop(columns=['Student ID'])
print("Columns after dropping 'Student ID':", df.columns.tolist())

Columns after dropping 'Student ID': ['Study Hours per Week', 'Attendance Rate', 'Previous Grades', 'Participation in Extracurricular Activities', 'Parent Education Level', 'Passed']


In [ ]:
df

,Study Hours per Week,Attendance Rate,Previous Grades,Participation in Extracurricular Activities,Parent Education Level,Passed
1,9.3,95.3,60.6,No,High School,No
3,17.6,76.8,62.4,Yes,Bachelor,No
4,8.8,89.3,72.7,No,Master,No
5,8.8,73.8,69.3,Yes,High School,Yes
6,17.9,38.6,93.6,No,Doctorate,Yes
...,...,...,...,...,...,...
1493,10.9,87.3,76.0,Yes,Master,No
1494,17.7,87.9,39.4,No,Master,No
1495,20.0,101.7,63.5,Yes,Doctorate,No
1496,20.3,100.1,68.9,No,High School,Yes


In [ ]:
# Select features and target
X = df[["Study Hours per Week", "Attendance Rate", "Previous Grades", "Participation in Extracurricular Activities", "Parent Education Level"]]
y = df["Passed"]

In [ ]:
X

,Study Hours per Week,Attendance Rate,Previous Grades,Participation in Extracurricular Activities,Parent Education Level
1,9.3,95.3,60.6,No,High School
3,17.6,76.8,62.4,Yes,Bachelor
4,8.8,89.3,72.7,No,Master
5,8.8,73.8,69.3,Yes,High School
6,17.9,38.6,93.6,No,Doctorate
...,...,...,...,...,...
1493,10.9,87.3,76.0,Yes,Master
1494,17.7,87.9,39.4,No,Master
1495,20.0,101.7,63.5,Yes,Doctorate
1496,20.3,100.1,68.9,No,High School


In [ ]:
y

,Passed
1,No
3,No
4,No
5,Yes
6,Yes
...,...
1493,No
1494,No
1495,No
1496,Yes


In [ ]:

# Define categorical and numeric features
categorical_features = [ "Participation in Extracurricular Activities", "Parent Education Level"]
numeric_features = ["Study Hours per Week", "Attendance Rate", "Previous Grades"]

In [ ]:
# Create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

In [ ]:
# Create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=42))
])

In [ ]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['Participation in '
                                                   'Extracurricular Activities',
                                                   'Parent Education Level']),
                                                 ('num', 'passthrough',
                                                  ['Study Hours per Week',
                                                   'Attendance Rate',
                                                   'Previous Grades'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [ ]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test, y_pred)

0.5296803652968036

In [ ]:
X_test.sample(5)

,Study Hours per Week,Attendance Rate,Previous Grades,Participation in Extracurricular Activities,Parent Education Level
299,13.1,84.3,60.4,Yes,Associate
1191,3.3,45.8,64.2,Yes,Master
42,9.4,125.8,59.9,Yes,Associate
1287,12.5,107.8,65.7,No,Doctorate
11,7.7,115.5,41.2,Yes,Master


In [ ]:
import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(pipeline, f)